In [15]:
import csv
import json
import pandas as pd
import numpy as np
from io import StringIO
import matplotlib.pyplot as plt
import os
import matplotlib.image as mpimg

# Default settings

## Figure

In [31]:
class Figure:
    def __init__(self, title=None, x_label=None, y_label=None, label_size=20, axis_label_size=16, elapsed_time=None):
        self.title = title
        self.x_label = x_label
        self.y_label = y_label
        self.label_size = label_size
        self.axis_label_size = axis_label_size
        self.elapsed_time = elapsed_time

    def plot(self, x, y):
        plt.figure(figsize=(14, 6))
        
        # Plot each temperature series dynamically
        for label, y_data in y.items():
            x_data = x[label] 
            plt.plot(x_data, y_data, label=label, linewidth=2)
            
        plt.title(self.title, fontsize=self.label_size)
        plt.xlabel(self.x_label, size=self.label_size)
        plt.ylabel(self.y_label, size=self.label_size)            
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.xlim(0,self.elapsed_time)
        plt.xticks(fontsize=self.axis_label_size)
        plt.yticks(fontsize=self.axis_label_size)
        plt.legend(loc='lower right',fontsize=self.axis_label_size)
        plt.show()

# Sensor

In [38]:
class Sensor_data_processor:
    def __init__(self, input_path, output_path):
        self.input_path = input_path
        self.output_path = output_path
        self.dfs = {}  # Added to safely track dataframes by test number

    def column_adjust(self, input_path, output_path, skip_row=0):
        col_name = ['Timestamp', 'Datetime', 'CH101 Temp_Fan_In', 'CH102 Temp_Fan Mid', 'CH103 Temp_Fan Out', 'CH104 Temp_Rad_In', 'CH105 Flow_In', 'CH106 Temp_Rad_Out', 'CH107 Flow_Out', 'CH108 Temp_Rad_Out', 'CH109 Temp_Amb', 'CH110 Temp_Air_Top', 'CH111 Temp_Air_Mid', 'CH112 Temp_Air_Bottom', 'CH113 P_Pump_In_V', 'CH114 P_Rad_In_V', 'CH115 P_Rad_Out_V', 'CH116 P_Res_In_V', 'CH117 P_Res_Out_V', 'CH118', 'CH119', 'CH120 ']
        df_sensor = pd.read_csv(input_path, sep=",", skiprows=skip_row)
        df_sensor.columns = col_name
        df_sensor.to_csv(output_path, sep=",", index=False)
        return df_sensor

    def sensor_flow(self, df_sensor):

        flow_columns = [col for col in df_sensor.columns if 'flow' in col.lower()]
        
        if flow_columns:
            for col in flow_columns:
                df_sensor[col] = pd.to_numeric(df_sensor[col], errors='coerce')

                # cLculate and append
                df_sensor_calculated= (5.1474 * df_sensor[col]) / (2 * 60)  + ( 68.27 / 60) 

                #check outlier option 1 add NaN 
                df_sensor[f'{col}_calculated'] = np.where( 
                    (df_sensor_calculated < 2.5 ) & (df_sensor_calculated > 1.3), 
                    df_sensor_calculated, 
                    np.nan)

                #check outlier option 2 drop row 
                # df_sensor = df_sensor[df_sensor_calculated < 200]

        else:
            print("Warning: No columns containing 'flow' were found.")
            
        return df_sensor

    def pressure(self, df_sensor):

        pressure_columns = [col for col in df_sensor.columns if 'p_' in col.lower() and '_V' in col]

        if pressure_columns:
            for col in pressure_columns:
                df_sensor[col] = pd.to_numeric(df_sensor[col], errors='coerce')
                
                # calculate and append
                df_sensor[f'{col}_In_Pa'] = 4/(3.5-0.5) * (df_sensor[col] - 0.5)
                

        else:
            print("Warning: No columns containing 'P_' were found.")
            
        return df_sensor

    def normalized_temp(self, df_sensor):
        normalized_temp_column = [col for col in df_sensor.columns if 'Temp' in col]

        if normalized_temp_column: 
            for col in normalized_temp_column:
                df_sensor[col]=pd.to_numeric(df_sensor[col], errors='coerce')

                df_sensor[f'{col}_Normalized'] = df_sensor[col] - df_sensor['CH109 Temp_Amb']

        return df_sensor               

    def complete_sensor_processing(self, data_path, test_number=None):
        output_path = self.output_path
        df_sensor = self.column_adjust(input_path=data_path, output_path=output_path)
        df_sensor = self.sensor_flow(df_sensor)
        df_sensor = self.pressure(df_sensor)
        df_sensor = self.normalized_temp(df_sensor)
        df_sensor.to_csv(output_path, sep=',', index=False)
        
        if test_number is not None:
            self.dfs[test_number] = df_sensor
            
        return df_sensor

    def evaluate_test_summary(self, test_number, df_sensor=None):
        
        if df_sensor is None:
            if test_number in self.dfs:
                df_sensor = self.dfs[test_number]
            else:
                # Fallback to reading the output_path file directly if needed
                df_sensor = pd.read_csv(self.output_path)

        summary = {}
        target_cols = [col for col in df_sensor.columns if 'Temp' in col or '_In_Pa' in col or 'flow' in col.lower()]
        
        for col in target_cols:
            summary[col] = {
                "mean": float(df_sensor[col].mean()),
                "std": float(df_sensor[col].std()),
                "min": float(df_sensor[col].min()),
                "max": float(df_sensor[col].max())
            }
        
        summary_df = pd.DataFrame(summary).T
        
        # Determine directory from output_path
        summary_dir = os.path.dirname(self.output_path)
        if summary_dir:
            os.makedirs(summary_dir, exist_ok=True)
            
        summary_path = os.path.join(summary_dir, f"Test{test_number}_summary.csv") if summary_dir else f"Test{test_number}_summary.csv"
        summary_df.to_csv(summary_path)
        return summary_df


class Sensor_plot:
    def __init__(self, data, elapsed_time, test_number, plot_output_folder):
        self.data = data
        self.elapsed_time = elapsed_time
        self.test_number = test_number
        self.plot_output_folder = plot_output_folder

    def plot_temp(self, columns):

        title = 'Temperature Profile for Test ' + str(self.test_number)
        fig = Figure( title=title, x_label='Time (s)', y_label='Temperature (°C)', elapsed_time=self.elapsed_time)
        x_series = {}; y_series = {}

        #s Series -> dictionaries
        for col in columns:
            if col in self.data.columns:
                valid_data = self.data.dropna(subset=['Timestamp', col])
                x_series[col] = valid_data['Timestamp']; y_series[col] = valid_data[col]
            else:
                print(f"Warning: Column '{col}' not found in the data.")

        # Plot all series inside one single figure
        if y_series:
            fig.plot(x_series, y_series)

        # Export 
        file_name = title + ".png"
        file_path = os.path.join(self.plot_output_folder, file_name)
        plt.savefig(file_path, dpi=700, bbox_inches="tight")
       
    def plot_pressure(self, columns):

        title = 'Pressure Data for Test' + self.test_number
        fig = Figure( title=title, x_label='Time (s)', y_label='Temperature (°C)', elapsed_time=self.elapsed_time)
        x_series = {}; y_series = {}

        for col in columns:
            if col in self.data.columns:
                valid_data = self.data.dropna(subset=['Timestamp', col]) #drop nan
                x_series[col] = valid_data['Timestamp']; y_series[col] = valid_data[col]
            else:
                print(f"Warning: Column '{col}' not found in the data.")

        if y_series:
                    fig.plot(x_series, y_series)

        file_name = title +".png"
        file_path = os.path.join(self.plot_output_folder, file_name)
        plt.savefig(file_path, dpi=700, bbox_inches= "tight")

    def plot_flow(self, columns):

        title = "Flow for Test" + self.test_number
        fig = Figure( title=title, x_label='Time (s)', y_label='Temperature (°C)', elapsed_time=self.elapsed_time)
        x_series = {}; y_series = {}
        
        for col in columns:
            if col in self.data.columns: 
                valid_data = self.data.dropna(subset=['Timestamp', col]) #fdrop nan
                x_series[col] = valid_data['Timestamp']; y_series[col] = valid_data[col]
            else: 
                print(f"Warning: Column{col} not found in the data.")

        if y_series:
            fig.plot(x_series, y_series)

        file_name = title +".png"
        file_path = os.path.join(self.plot_output_folder, file_name)
        plt.savefig(file_path, dpi=700, bbox_inches= "tight")


# MTC

In [ ]:
class MTC_data_processor:
    def __init__(self, input_path, output_path):
        self.input_path = input_path
        self.output_path = output_path

    def clean_and_process_log(self, input_path, skip_row=1):
        column_names = ["Date","Time (ms)","Unknown","Bash Command","Bash Received","Bash_temp","Echo_Time","Status"]

        try:
            with open(input_path, "r", encoding="utf-8") as file:
                lines = file.readlines()

            lines = lines[skip_row:]
            cleaned_lines = []
            for line in lines:
                stripped_line = line.strip()
                if stripped_line:
                    cleaned_lines.append(stripped_line + "\n")

            raw_text_data = "".join(cleaned_lines)

            df = pd.read_csv(StringIO(raw_text_data), sep=";", names=column_names, header=None, engine="python", on_bad_lines="skip",)

            df.columns = df.columns.str.strip()

            if "Bash_temp" in df.columns:
                df["Bash_temp"] = pd.to_numeric(df["Bash_temp"].astype(str).str.strip(), errors="coerce") / 10

            print(f"Successfully cleaned '{input_path}'.")
            return df

        except FileNotFoundError:
            print(f"Error: The file '{input_path}' was not found.")
        except Exception as exc:
            print(f"An error occurred: {exc}")
            return None

    def cleaned_datetime(self, data):
        data["Date"] = pd.to_datetime(data["Date"], format="%H:%M:%S.%f", errors="coerce")
        data = data.dropna(subset=["Date"]).copy()

        time_diffs = data["Date"].diff().dt.total_seconds()
        midnight_masks = time_diffs < -43200
        days_to_add = midnight_masks.cumsum()
        data["Date"] = data["Date"] + pd.to_timedelta(days_to_add, unit="D")

        data = data.sort_values(by="Date").reset_index(drop=True)
        start_time = data["Date"].iloc[0]
        data["Elapsed_Time"] = (data["Date"] - start_time).dt.total_seconds()

        return data

    def power_calculation(self, data):
        # State dictionaries to store the latest known RTC and RAP values for devices 1 to 6
        latest_rtc = {x: 0.0 for x in range(1, 7)}
        latest_rap = {x: 0.0 for x in range(1, 7)}
        
        new_rows = []
    
        for i, row in data.iterrows():
            cmd = str(row['Bash Received']).strip().lower() if 'Bash Received' in data.columns else ""
            val = float(row['Bash_temp']) if 'Bash_temp' in data.columns else 0.0

            # 1. Update state dictionaries FIRST so any incoming RTC/RAP is recorded
            for x in range(1, 7):
                if cmd == f"{x}rtc":
                    latest_rtc[x] = val
                elif cmd == f"{x}rap":
                    latest_rap[x] = val

            # 2. Append the current row
            new_rows.append(row)

            # 3. If the current row is an RTC command, append the PWR row immediately after it
            for x in range(1, 7):
                if cmd == f"{x}rtc":
                    xrtc = latest_rtc[x]
                    xrap = latest_rap[x]
                    
                    # Calculate power: 24 * rtc * (rap / 100)
                    xpwr_val = 24 * xrtc * (xrap / 100)

                    pwr_row = row.copy()
                    pwr_row['Bash Command'] = f"{x}PWR"
                    pwr_row['Bash Received'] = f"{x}pwr"
                    pwr_row['Bash_temp'] = xpwr_val  
                    pwr_row['Status'] = f"calculated pwr ({x}): {xpwr_val}"
                    
                    new_rows.append(pwr_row)
                    break
                
        result_df = pd.DataFrame(new_rows)
        # Re-sort and re-calculate Elapsed_Time so appended power rows fit seamlessly into the timeline
        return self.cleaned_datetime(result_df)

    def export_data(self, df, format_type="csv"):
        """Dedicated function to export the DataFrame to CSV or JSON."""
        if df is None or df.empty:
            raise ValueError("No data provided to export.")

        format_type = format_type.lower()
        if format_type == "csv":
            df.to_csv(self.output_path, sep=',', index=False, encoding='utf-8')
            print(f"Successfully exported data to CSV: '{self.output_path}'")
        elif format_type == "json":
            results_data = df.to_dict(orient="records")
            with open(self.output_path, mode="w", encoding="utf-8") as json_file:
                json.dump(results_data, json_file, indent=4)
            print(f"Successfully exported data to JSON: '{self.output_path}'")
        else:
            raise ValueError(f"Unsupported format type: {format_type}. Use 'csv' or 'json'.")
        
        
class MTC_plotter:
    def __init__(self, data, device_rat, device_rtc, device_rap, device_pwr, elapsed_time, test_number,plot_output_folder):
        self.data = data
        self.device_rat = device_rat
        self.device_rtc = device_rtc
        self.device_rap = device_rap
        self.device_pwr = device_pwr
        self.elapsed_time = elapsed_time
        self.test_number = test_number
        self.plot_output_folder = plot_output_folder

    def plot_all_devices(self):
        
        plot_data = self.data.copy()
        plot_data["Elapsed_Time"] = pd.to_numeric(plot_data["Elapsed_Time"], errors="coerce")
        plot_data["Bash_temp"] = pd.to_numeric(plot_data["Bash_temp"], errors="coerce")

        plt.figure(figsize=(14, 7))
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

        for ch in range(1, 7):
            rat_df = plot_data[
                plot_data["Bash Received"].astype(str).str.contains(self.device_rat, case=False)
                & plot_data["Bash Received"].astype(str).str.contains(str(ch))
            ].copy()

            if not rat_df.empty:
                plt.plot(rat_df["Elapsed_Time"],rat_df["Bash_temp"],label=f"Actual temperature device {ch} (RAT)",
                    linestyle="-",
                    color=colors[ch - 1],
                    alpha=0.9,
                )

        rtt_df = plot_data[
            plot_data["Bash Received"].astype(str).str.contains("1rtt", case=False)
        ].copy()

        if not rtt_df.empty:
            plt.plot(
                rtt_df["Elapsed_Time"],
                rtt_df["Bash_temp"],
                label="Target temperature (RTT)",
                linestyle="--",
                color="blue",
                alpha=0.7,
            )

        title = "RAT&RTT profile for Test" + self.test_number
        plt.title(title, size=20)
        plt.xlabel("Time (s)", size=20)
        plt.ylabel("Temperature (°C)", size=20)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend(loc="lower right", fontsize=16)
        plt.xticks(fontsize=16)
        plt.yticks(fontsize=16)
        plt.xlim(0,self.elapsed_time)
        # plt.ylim(-7,8)
 
        file_name = title +".png"
        file_path = os.path.join(self.plot_output_folder, file_name)
        plt.savefig(file_path, dpi=700, bbox_inches= "tight")

        plt.show()

    def plot_rtc(self):
    
        plot_data = self.data.copy()
        plot_data["Elapsed_Time"] = pd.to_numeric(plot_data["Elapsed_Time"], errors="coerce")
        plot_data["Bash_temp"] = pd.to_numeric(plot_data["Bash_temp"], errors="coerce")

        plt.figure(figsize=(14, 7))
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

        for ch in range(1, 7):
            rtc_df = plot_data[
                plot_data["Bash Received"].astype(str).str.contains(self.device_rtc, case=False)
                & plot_data["Bash Received"].astype(str).str.contains(str(ch))
            ].copy()

            if not rtc_df.empty:
                plt.plot(rtc_df["Elapsed_Time"],rtc_df["Bash_temp"],label=f"Current @ device {ch} (RTC)",marker=".",linestyle="-",
                    alpha=0.7,color=colors[ch - 1],)

        title = "RTC profile for Test" + self.test_number
        plt.title(title, size=20)
        plt.xlabel("Time (s)", size=20)
        plt.ylabel("Current (A)", size=20)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend(loc="lower right", fontsize=16)
        plt.xticks(fontsize=16)
        plt.yticks(fontsize=16)
        plt.xlim(0,self.elapsed_time)

        file_name = title +".png"
        file_path = os.path.join(self.plot_output_folder, file_name)
        plt.savefig(file_path, dpi=700, bbox_inches= "tight")

        plt.show()

    def plot_rap(self):
        plot_data = self.data.copy()

        plt.figure(figsize=(14, 7))
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

        for ch in range(1, 7):
            rap_df = plot_data[
                plot_data["Bash Received"].astype(str).str.contains(self.device_rap, case=False)
                & plot_data["Bash Received"].astype(str).str.contains(str(ch))
            ].copy()

            if not rap_df.empty:
                plt.plot(rap_df["Elapsed_Time"],rap_df["Bash_temp"], label=f"PWM @ device {ch} (RAP)", marker=".", linestyle="-", alpha=0.7, color=colors[ch - 1])

        title = "RAP profile for Test"  + self.test_number
        plt.title(title, size=20)
        plt.xlabel("Time (s)", size=20)
        plt.ylabel("PWM (%)", size=20)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend(loc="lower right", fontsize=16)
        plt.xticks(fontsize=16)
        plt.yticks(fontsize=16)
        plt.xlim(0,self.elapsed_time)

        file_name = title +".png"
        file_path = os.path.join(self.plot_output_folder, file_name)
        plt.savefig(file_path, dpi=700, bbox_inches= "tight")

        plt.show()

    def plot_power(self):
        plot_data = self.data.copy()

        plt.figure(figsize=(14, 7))
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

        for ch in range(1, 7):
            pwr_df = plot_data[
                plot_data["Bash Received"].astype(str).str.contains(self.device_pwr, case=False)
                & plot_data["Bash Received"].astype(str).str.contains(str(ch))
            ].copy()

            if not pwr_df.empty:
                # Fixed: Changed Elapsed_time to Elapsed_Time and corrected linestyle to '-'
                plt.plot(pwr_df["Elapsed_Time"], pwr_df["Bash_temp"], label=f"Power @ device {ch}", marker=".", linestyle="-", alpha=0.7, color=colors[ch - 1])

        title = "Power profile for Test" + self.test_number
        plt.title(title, size=20)
        plt.xlabel("Time (s)", size=20)
        plt.ylabel("Power (W)", size=20)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend(loc="lower right", fontsize=16)
        plt.xticks(fontsize=16)
        plt.yticks(fontsize=16)
        plt.xlim(0, self.elapsed_time)

        file_name = title +".png"
        file_path = os.path.join(self.plot_output_folder, file_name)
        plt.savefig(file_path, dpi=700, bbox_inches= "tight")

        plt.show()

# if __name__ == "__main__":

    # input_path = "MTC_Test1.txt"
    # output_path = "processed_" + input_path.replace(".txt", ".csv")
    # elapsed_time = 3600
    
    # device_rat = "1rat"
    # device_rtc = "1rtc"
    # device_rap = "1rap"
    # device_pwr = "1pwr"

    # processor = MTC_data_processor(input_path=input_path, output_path=output_path)
    
    # # 1. Clean, finalize initial log data
    # df = processor.clean_and_process_log(input_path)
    # df_cleaned_datetime = processor.cleaned_datetime(df)
    
    # # 2. Perform power calculation on finalized data so power rows get accurate timestamps
    # df_processed_with_power = processor.power_calculation(df_cleaned_datetime)
    
    # # 3. Export the comprehensive dataset with power included to CSV
    # processor.export_data(df_processed_with_power, format_type="csv")
    
    # # 4. Plot using the fully processed dataset containing power records
    # df_plot = MTC_plotter(df_processed_with_power, device_rat=device_rat, device_rtc=device_rtc, device_rap=device_rap, device_pwr=device_pwr, elapsed_time=elapsed_time)

    # df_plot.plot_all_devices()
    # df_plot.plot_rtc()
    # df_plot.plot_rap()
    # df_plot.plot_power()
    